# Mission 3 — 2-5: 가중 TF-IDF와 기존 앙상블 결합

목표: 내부 검증 Macro F1 0.66, **예측 임계값 0.5 고정**.
기존 2-4 B 앙상블(0.647423)에 TF-IDF의 어휘 정보를 더합니다.
이전 50:50 TF-IDF 결합은 성능이 낮았으므로 이번에는 **B 0.8 + TF-IDF 0.2**로 사전 고정합니다.
이 비율은 확인된 최적 비율이 아니라 이번 실험의 가설입니다. 비율 탐색은 하지 않습니다.

동일 TF-IDF 입력·특징·분할·C에서 로지스틱 회귀의 클래스 가중치만 비교합니다.
- T0: 클래스 가중치 없음.
- T1: 학습 데이터에서만 계산한 양성 가중치 `clip(sqrt(negative/positive),1,3)`.
- 주 비교: B 80% + T0 20% 대 B 80% + T1 20%. B 단독과 T0/T1 단독도 기록합니다.

새 학습은 TF-IDF와 로지스틱 회귀 9개 × 2조건입니다. RoBERTa 재학습은 없습니다.
CPU 런타임에서 실행하며 TF-IDF 특징 때문에 메모리 사용량이 클 수 있습니다.
클래스별 모델은 완료할 때마다 `/content`에 저장합니다. 같은 VM에서 재실행하면 완료 라벨을 재사용합니다.
VM 자체가 삭제되면 해당 파일은 사라집니다.

업로드 파일:
1. `mission3_train_json.zip`
2. `m3_targeted_weight_results.tar.gz` (기존 내부 분할)
3. `m3_2_4_fixed_ensemble_results.tar` (기존 B 예측)

공식 Validation을 사용하지 않습니다. 이 내부 검증셋을 반복 사용한 탐색 실험으로,
점수가 올라도 독립 최종 성능으로 해석하지 않습니다.


## 1. 입력과 환경 확인


In [ ]:
from pathlib import Path
import json, tarfile, zipfile, io, hashlib, platform, csv, warnings
import importlib.metadata as im
import numpy as np
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import precision_recall_fscore_support, f1_score
from google.colab import files

ROOT=Path('/content')
TRAIN_ZIP=ROOT/'mission3_train_json.zip'
OLD=ROOT/'m3_targeted_weight_results.tar.gz'
ENSEMBLE=ROOT/'m3_2_4_fixed_ensemble_results.tar'
OUT=ROOT/'m3_2_5_tfidf_weight'
OUT.mkdir(exist_ok=True)
TARGETS=['고열','구토','두통','복통','어지러움','열상','오심','전신쇠약','호흡곤란']
for p in [TRAIN_ZIP,OLD,ENSEMBLE]:
    assert p.is_file(), f'업로드 필요: {p}'
def sha(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
plan={'threshold':.5,'B_weight':.8,'tfidf_weight':.2,'C':2.,'seed':42,
      'max_iter':1000,'solver':'liblinear','analyzer':'char','ngram_range':[2,5],
      'min_df':2,'max_features':150000,'sublinear_tf':True,
      'positive_weight_rule':'clip(sqrt(negative/positive),1,3)',
      'input_sha256':{p.name:sha(p) for p in [TRAIN_ZIP,OLD,ENSEMBLE]},
      'environment':{'python':platform.python_version(),**{k:im.version(k) for k in ['numpy','pandas','scikit-learn','joblib']}}}
config_path=OUT/'experiment_plan.json'
if config_path.exists():
    assert json.loads(config_path.read_text())==plan, '이전 설정·입력·환경과 다릅니다. 새 출력 폴더를 사용하세요.'
else:config_path.write_text(json.dumps(plan,ensure_ascii=False,indent=2))
print('고정 비율 B 0.8 / TF-IDF 0.2、threshold 0.5')


## 2. 원래 분할 복원과 B 예측 정렬 검사


In [ ]:
with tarfile.open(OLD,'r:*') as t:
    manifest=t.extractfile('split_manifest.csv').read()
with tarfile.open(ENSEMBLE,'r:*') as t:
    raw=t.extractfile('B_base3_large_equal_predictions.npz').read()
with np.load(io.BytesIO(raw),allow_pickle=False) as z:
    base={k:z[k].copy() for k in ['probs','y_true','file_names','targets']}
assert list(base['targets'])==TARGETS
assert base['probs'].shape==base['y_true'].shape==(5792,9)
assert np.isfinite(base['probs']).all() and ((base['probs']>=0)&(base['probs']<=1)).all()
rows=[]
with zipfile.ZipFile(TRAIN_ZIP) as z:
    for name in sorted(z.namelist()):
        p=Path(name)
        if p.suffix.lower()!='.json' or '__MACOSX' in p.parts or p.name.startswith('._'):continue
        d=json.loads(z.read(name))
        text=' '.join(u.get('text','') for u in d['utterances'])
        assert isinstance(d['symptom'],list)
        rows.append({'file_name':p.name,'text':text,**{k:int(k in d['symptom']) for k in TARGETS}})
df=pd.DataFrame(rows);split=pd.read_csv(io.BytesIO(manifest))
assert len(df)==29200 and not df.file_name.duplicated().any()
assert not split.file_name.duplicated().any() and set(split.file_name)==set(df.file_name)
assert set(split.split)=={'train','dev_valid'}
ordered=split.merge(df,on='file_name',how='left',validate='one_to_one',sort=False)
train=ordered[ordered.split=='train'].reset_index(drop=True)
valid=ordered[ordered.split=='dev_valid'].reset_index(drop=True)
assert (len(train),len(valid))==(23408,5792)
assert np.array_equal(base['file_names'],valid.file_name.to_numpy())
assert np.array_equal(base['y_true'],valid[TARGETS].to_numpy())
y_train=train[TARGETS].to_numpy();y=valid[TARGETS].to_numpy()
baseline_score=float(f1_score(y,base['probs']>=.5,average='macro',zero_division=0))
assert abs(baseline_score-0.6474228567088907)<1e-9, '2-4 B 파일을 확인하세요.'
(OUT/'source_provenance.json').write_text(json.dumps({'manifest_sha256':hashlib.sha256(manifest).hexdigest(),
    'B_prediction_sha256':hashlib.sha256(raw).hexdigest(),'B_macro_f1':baseline_score,
    'train_count':len(train),'valid_count':len(valid)},indent=2))
print('분할·정답 일치, B Macro F1:',baseline_score)


## 3. TF-IDF 학습 — 내부 train에만 fit


In [ ]:
vectorizer=TfidfVectorizer(analyzer='char',ngram_range=(2,5),min_df=2,
    max_features=150000,sublinear_tf=True,dtype=np.float32)
X_train=vectorizer.fit_transform(train.text)
X_valid=vectorizer.transform(valid.text)
model_dir=OUT/'models';model_dir.mkdir(exist_ok=True)
joblib.dump(vectorizer,model_dir/'vectorizer.joblib')
positive=y_train.sum(0);negative=len(train)-positive
assert ((positive>0)&(negative>0)).all()
weights=np.clip(np.sqrt(negative/positive),1.,3.)
pd.DataFrame({'symptom':TARGETS,'positive':positive,'negative':negative,
              'T1_positive_weight':weights}).to_csv(OUT/'training_weights.csv',index=False)
print('TF-IDF:',X_train.shape, X_valid.shape)


## 4. T0·T1 학습과 클래스별 재개

조건별 C·solver·seed는 동일합니다. T1에서 class_weight만 바꿉니다. 수렴 경고가 나오면 결과를 확정하지 않고 중단합니다.


In [ ]:
tfidf_probs={}
for name in ['T0_unweighted','T1_weighted']:
    probs=np.zeros((len(valid),len(TARGETS)),dtype=np.float64)
    for j,target in enumerate(TARGETS):
        path=model_dir/f'{name}_label{j}.joblib'
        if path.exists():
            clf=joblib.load(path)  # 이 노트북이 현재 OUT에 저장한 모델만 읽습니다.
            print('완료 모델 재사용:',name,target)
        else:
            cw=None if name=='T0_unweighted' else {0:1.,1:float(weights[j])}
            clf=LogisticRegression(C=2.,max_iter=1000,solver='liblinear',random_state=42,class_weight=cw)
            with warnings.catch_warnings():
                warnings.simplefilter('error',ConvergenceWarning)
                clf.fit(X_train,y_train[:,j])
            temp=path.with_suffix('.tmp')
            joblib.dump(clf,temp);temp.replace(path)
            print('학습 완료:',name,target)
        assert list(clf.classes_)==[0,1]
        probs[:,j]=clf.predict_proba(X_valid)[:,1]
    assert np.isfinite(probs).all() and ((probs>=0)&(probs<=1)).all()
    tfidf_probs[name]=probs
print('T0/T1 학습 완료')


## 5. 고정 비율·고정 임계값 비교


In [ ]:
predictions={'B_reference':base['probs'],**tfidf_probs,
    'B80_T0_20':.8*base['probs']+.2*tfidf_probs['T0_unweighted'],
    'B80_T1_20':.8*base['probs']+.2*tfidf_probs['T1_weighted']}
summary=[];labels=[]
for name,probs in predictions.items():
    pred=probs>=.5
    score=float(f1_score(y,pred,average='macro',zero_division=0))
    summary.append({'experiment':name,'macro_f1':score,'delta_vs_B':score-baseline_score,
                    'threshold':.5,'target_reached':score>=.66})
    p,r,f,support=precision_recall_fscore_support(y,pred,average=None,zero_division=0)
    for j,target in enumerate(TARGETS):
        labels.append({'experiment':name,'symptom':target,'precision':p[j],'recall':r[j],
                       'f1':f[j],'support':int(support[j]),'predicted_positive':int(pred[:,j].sum())})
    if name!='B_reference':
        np.savez_compressed(OUT/f'{name}_predictions.npz',probs=probs,y_true=y,
                            file_names=base['file_names'],targets=np.asarray(TARGETS))
summary=pd.DataFrame(summary);labels=pd.DataFrame(labels)
summary.to_csv(OUT/'summary.csv',index=False);labels.to_csv(OUT/'per_label.csv',index=False)
display(summary);display(labels[labels.symptom.isin(['오심','두통','전신쇠약'])])
scores=summary.set_index('experiment').macro_f1
print('클래스 가중치 효과(결합 기준):',scores['B80_T1_20']-scores['B80_T0_20'])
print('기존 B 대비 개선:',scores['B80_T1_20']-baseline_score)


## 6. 결과 TAR 다운로드


In [ ]:
lines=['# 2-5 가중 TF-IDF 결과','',
    '내부 검증 5,792건 / threshold 0.5 / B 80% + TF-IDF 20% 사전 고정.',
    '동일 학습 조건에서 T0(비가중)와 T1(양성 sqrt 가중)을 비교했다.', '',
    '|실험|Macro F1|B 대비|','|---|---:|---:|']
for row in summary.to_dict('records'):
    lines.append(f"|{row['experiment']}|{row['macro_f1']:.6f}|{row['delta_vs_B']:+.6f}|")
lines+=['','공식 Validation 결과가 아니다. 반복 사용한 내부 검증셋의 탐색 결과다.',
        'TF-IDF 모델은 별도 models 폴더에 저장된다. 이 결과 TAR에는 모델 가중치가 없다.',
        '예측 npz에는 통화 파일명이 있으므로 공개 Git에는 집계 지표만 공유한다.']
(OUT/'REPORT.md').write_text('\n'.join(lines),encoding='utf-8')
destination=ROOT/'m3_2_5_tfidf_weight_results.tar'
with tarfile.open(destination,'w') as archive:
    for p in sorted(OUT.iterdir()):
        if p.is_file():archive.add(p,arcname=p.name)
files.download(str(destination))


## 7. 선택: 학습된 TF-IDF 모델 다운로드

최종 채택 시 새 데이터 추론에 필요합니다. 결합에 쓰는 RoBERTa 4개 가중치는 별도입니다.


In [ ]:
DOWNLOAD_TFIDF_MODELS=False
if DOWNLOAD_TFIDF_MODELS:
    destination=ROOT/'m3_2_5_tfidf_models.tar'
    with tarfile.open(destination,'w') as archive:archive.add(model_dir,arcname='models')
    files.download(str(destination))
